# Project 1: AI-Powered Emotion Detection from Text
This notebook demonstrates an end-to-end NLP pipeline to classify the emotional tone of text. We will use the `dair-ai/emotion` dataset from Hugging Face, preprocess it, and train a Logistic Regression model using TF-IDF features.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import nltk

import warnings
warnings.filterwarnings('ignore')

## 1. Dataset Collection
We will load the `dair-ai/emotion` dataset. It consists of English Twitter messages with six basic emotions: anger, fear, joy, love, sadness, and surprise.

In [ ]:
# Load dataset from Hugging Face
dataset = load_dataset("dair-ai/emotion", "split")
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

label_mapping = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
train_df['label_name'] = train_df['label'].map(label_mapping)
test_df['label_name'] = test_df['label'].map(label_mapping)

train_df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=train_df, x='label_name', order=train_df['label_name'].value_counts().index)
plt.title('Distribution of Emotion Classes in Training Data')
plt.show()

## 3. Text Preprocessing
We will clean the text (lowercasing, special char removal) and apply tokenization, stopword removal, and lemmatization using NLTK.

In [ ]:
from data_preprocessing import preprocess_text

# Apply preprocessing to train and test sets
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)

train_df[['text', 'clean_text']].head()

## 4. Feature Extraction (TF-IDF)
We convert the cleaned text into numerical vectors using `TfidfVectorizer`.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['clean_text'])
y_train = train_df['label']

X_test = vectorizer.transform(test_df['clean_text'])
y_test = test_df['label']

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

## 5. Model Building
We will train a Logistic Regression classifier.

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)
print("Model trained successfully!")

## 6. Model Evaluation
Evaluate the model using Accuracy, F1-score, and a Confusion Matrix.

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=[label_mapping[i] for i in range(6)]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[label_mapping[i] for i in range(6)], yticklabels=[label_mapping[i] for i in range(6)])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 7. Make Predictions on New Text
Let's test our model with some custom inputs.

In [ ]:
def predict_emotion(text):
    cleaned = preprocess_text(text)
    features = vectorizer.transform([cleaned])
    pred = model.predict(features)[0]
    return label_mapping[pred]

samples = [
    "I am so excited to learn Natural Language Processing!",
    "This was the worst movie I have ever seen.",
    "I'm terrified of what might happen next.",
    "I am completely amazed and shocked by this news!"
]

for sample in samples:
    print(f"Text: '{sample}'\nPredicted Emotion: {predict_emotion(sample).upper()}\n")